In [ ]:
# Install Sentence Transformers
!pip install -q sentence-transformers

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import os
import torch

# Paths
BASE_PATH = '/content/drive/MyDrive/imdb_data/ekko_artifacts'
METADATA_PATH = f'{BASE_PATH}/metadata_enriched.parquet'
OUTPUT_PATH = f'{BASE_PATH}/ai_embeddings.npy'

print("Libraries installed and Drive mounted.")

Libraries installed and Drive mounted.


In [ ]:
# Load the clean metadata
print("Loading metadata...")
df = pd.read_parquet(METADATA_PATH)

# Check if text_blob exists
if 'text_blob' not in df.columns:
    raise ValueError("Error: 'text_blob' column missing. Did you run Notebook 01 correctly?")

print(f"Loaded {len(df)} items.")

# Load the AI Model
# 'all-mpnet-base-v2' is fast and accurate.
# If you want slightly lower accuracy but 2x faster, change to 'all-MiniLM-L6-v2'
MODEL_NAME = 'all-mpnet-base-v2'
print(f"Loading Model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on {device.upper()}")

Loading metadata...
Loaded 428167 items.
Loading Model: all-mpnet-base-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded on CUDA


In [ ]:
# Prepare the text list
sentences = df['text_blob'].tolist()

# Encode in Batches
print("Starting Embedding Generation... (This may take a while)")

# Batch size 128 is usually safe for T4 GPU
embeddings = model.encode(
    sentences,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True # Important for Cosine Similarity later
)

print(f"Embeddings Shape: {embeddings.shape}")
# Should be (428167, 384)

print(f"Saving embeddings to: {OUTPUT_PATH}")
np.save(OUTPUT_PATH, embeddings)

print("DONE!")

Starting Embedding Generation... (This may take a while)


Batches:   0%|          | 0/3346 [00:00<?, ?it/s]

Embeddings Shape: (428167, 768)
Saving embeddings to: /content/drive/MyDrive/imdb_data/ekko_artifacts/ai_embeddings.npy
DONE!


Batches: 100%
 3346/3346 [1:17:25<00:00,  2.60it/s]
